In [ ]:
import pickle
import sys
import CRPS.CRPS as pscore
import numpy as np
from pathlib import Path


import multiprocessing as mp
mp.set_start_method('spawn')

sys.path.insert(0, '../LSTM_next_activity_duration/notebooks/evaluation/')
sys.path.insert(0, '../../../../Evaluation')

import conduct_evaluation
import normal_evaluation.normal_evaluation
from prefix_duration_predictor import PrefixDurationPredictor, NOTEBOOK_DIR
from normal_evaluation.lstm_evaluation import SampleOutcomes_LSTM


get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [ ]:
with open('../../../transformed_event_logs/BPIC_2017_all_test.pickle', 'rb') as f:
    test_data = pickle.load(f)


n_processes = 20
batch_size = 10
N = 1000

In [3]:

event_log_properties = {
    'case_name' : 'case:concept:name',
    'concept_name' : 'concept:name',
    'timestamp_name' : 'time:timestamp_start',
    'time_since_case_start_column' : '',
    'time_since_last_event_column' : '',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 1,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.0,
    'window_size' : 'auto',
    'categorical_columns' : ['concept:name', 'org:resource_start'],
    'continuous_columns' : ['seconds_in_day', 'day_in_week', 'duration_seconds'],
    'continuous_positive_columns' : []
}

#NOTEBOOK_DIR = Path(__file__).resolve().parent
LSTM_ROOT = (NOTEBOOK_DIR / "../..").resolve()
LOADER_DIR = (NOTEBOOK_DIR / "../../../../load/event_log_loader").resolve()
ENCODED_DIR = (NOTEBOOK_DIR / "../../../../load/encoded_data").resolve()
TRANSFORMED_LOG_DIR = (NOTEBOOK_DIR / "../../../../../transformed_event_logs").resolve()
MODEL_DIR = (NOTEBOOK_DIR / "../training_variational_dropout/BPIC17").resolve()

TRAIN_DATA_PATH = (ENCODED_DIR / "BPIC_2017_all_1_train.pkl").resolve()
TRAIN_EVENT_LOG_PATH = (TRANSFORMED_LOG_DIR / "BPIC_2017_all_train.csv").resolve()

selected_cat_attributes = ['concept:name', 'org:resource_start']
selected_num_attributes = ['seconds_in_day', 'day_in_week']

lstm_predictor = PrefixDurationPredictor(
        train_loader_path = TRAIN_DATA_PATH,
        train_event_log_path = TRAIN_EVENT_LOG_PATH,
        model_dir = MODEL_DIR,
        model_path = None,
        event_log_properties= event_log_properties,
        selected_cat_attributes = selected_cat_attributes,
        selected_num_attributes = selected_num_attributes,
        device = 'cpu'
)

Embeddings:  ModuleList(
  (0): Embedding(43, 16)
  (1): Embedding(149, 16)
)
Total embedding feature size:  32
Input feature size:  34
Cells hidden size:  128
Number of LSTM layer:  2
Dropout rate:  0.1




In [ ]:
evaluator_A = conduct_evaluation.ConductEvaluation(lstm_predictor, SampleOutcomes_LSTM, {
                                                    },
                                    test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|          | 0/6068 [00:00<?, ?it/s]

  0%|          | 0/6068 [00:10<?, ?it/s]Process ForkPoolWorker-28:
Process ForkPoolWorker-10:
  0%|          | 0/6068 [01:15<?, ?it/s]Process ForkPoolWorker-11:
Process ForkPoolWorker-18:
Process ForkPoolWorker-19:
Process ForkPoolWorker-2:
Process ForkPoolWorker-3:
Process ForkPoolWorker-13:
Process ForkPoolWorker-9:
Process ForkPoolWorker-31:
Process ForkPoolWorker-7:
Process ForkPoolWorker-5:
Process ForkPoolWorker-16:

Process ForkPoolWorker-25:
Process ForkPoolWorker-27:
Process ForkPoolWorker-6:
Process ForkPoolWorker-15:
Process ForkPoolWorker-21:
Process ForkPoolWorker-4:
Process ForkPoolWorker-8:
Process ForkPoolWorker-17:
Process ForkPoolWorker-30:
Process ForkPoolWorker-26:
Process ForkPoolWorker-12:
Process ForkPoolWorker-24:
Process ForkPoolWorker-32:
Process ForkPoolWorker-29:
Traceback (most recent call last):


ConnectionResetError: [Errno 104] Connection reset by peer

Process ForkPoolWorker-20:
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkPoolWorker-22:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkPoolWorker-1:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/LordKunkler/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/home/LordKunkler/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

In [ ]:
np.mean(get_pscores(likelihoods_A))